# ULTRON v0.3 — training
Run the cells from top to bottom. This trains the custom ULTRON Transformer from scratch on the project dataset.

In [ ]:
!rm -rf /content/Ultron-ai
!git clone -b ultron-brain-v0.1 https://github.com/yosimaracabal2013-stack/Ultron-ai.git /content/Ultron-ai
%cd /content/Ultron-ai
!pip -q install torch


In [ ]:
import os, sys
sys.path.insert(0, '/content/Ultron-ai/ai')
os.environ['ULTRON_GPU_STEPS']='5000'
os.environ['ULTRON_CPU_STEPS']='500'
!python ai/train_v0_3.py


In [ ]:
# Quantize the trained checkpoint to INT8 for compact deployment.
import torch, json
from pathlib import Path
ckpt=Path('/content/Ultron-ai/ai/checkpoints/ultron_v0_3.pt')
out=Path('/content/Ultron-ai/ai/checkpoints/ultron_v0_2.pt')
data=torch.load(ckpt,map_location='cpu',weights_only=False)
q={}; scales={}
for name,t in data['model'].items():
    if torch.is_floating_point(t):
        scale=float(t.abs().max().item()/127.0) or 1e-8
        q[name]=torch.clamp(torch.round(t/scale),-127,127).to(torch.int8)
        scales[name]=scale
    else:
        q[name]=t
packed={'format':'ultron-int8-v1','model_int8':q,'scales':scales,'config':data['config']}
torch.save(packed,out)
print('INT8 checkpoint:',out)
print('size MB:',out.stat().st_size/1024/1024)


In [ ]:
from google.colab import files
files.download('/content/Ultron-ai/ai/checkpoints/ultron_v0_2.pt')
files.download('/content/Ultron-ai/ai/checkpoints/tokenizer_v0_3.json')


After training, the two downloaded files are the new ULTRON brain files: `ultron_v0_3_int8.pt` and `tokenizer_v0_3.json`.

In [ ]:
# Quick local generation test before deployment
import torch, sys
sys.path.insert(0,'/content/Ultron-ai/ai')
from tokenizer import CharTokenizer
from model import UltronTransformer
tok=CharTokenizer.load('/content/Ultron-ai/ai/checkpoints/tokenizer_v0_3.json')
raw=torch.load('/content/Ultron-ai/ai/checkpoints/ultron_v0_3.pt',map_location='cpu',weights_only=False)
model=UltronTransformer(**raw['config']); model.load_state_dict(raw['model']); model.eval()
p='User: What is your purpose?\
ULTRON:'
ids=torch.tensor([[tok.stoi.get(c,tok.stoi.get(' ')) for c in p]],dtype=torch.long)
with torch.no_grad(): out=model.generate(ids,80,temperature=0.35,top_k=10)[0].tolist()
print(tok.decode(out[len(ids[0]):]))
